## <span style="color:red; font-weight:bold;">Using Mixtral-8x7B-v0.1</span>

In [5]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "mistralai/Mixtral-8x7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(model_id)

text = "Hello my name is"
inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

model-00007-of-00019.safetensors:  29%|##8       | 1.41G/4.90G [00:00<?, ?B/s]

C:\Users\devanshi\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\devanshi\.cache\huggingface\hub\models--mistralai--Mixtral-8x7B-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model-00008-of-00019.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00009-of-00019.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00010-of-00019.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

...

model-00015-of-00019.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00016-of-00019.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00017-of-00019.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00018-of-00019.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00019-of-00019.safetensors:   0%|          | 0.00/4.22G [00:00<?, ?B/s]

In [3]:
# Function to generate text with the model
def generate_response(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate text
    with torch.no_grad():  # Disable gradient calculation for inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,       # Control randomness (lower = more deterministic)
            top_p=0.9,             # Nucleus sampling parameter
            repetition_penalty=1.2 # Discourage repetition
        )
        # Decode and return the generated text
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated_text

In [5]:
prompt = "Hello, I'm Dev! Who are you?"
response = generate_response(prompt)
print(response)

NameError: name 'tokenizer' is not defined

In [6]:
prompt = "WHy did the French revolution happen?"
response = generate_response(prompt)
print(response)

NameError: name 'tokenizer' is not defined

In [7]:
prompt = "Tell me about the political and imperial history of Russia"
response = generate_response(prompt)
print(response)

NameError: name 'tokenizer' is not defined

In [8]:
prompt = "Who are the most well-known pop singers of this year?"
response = generate_response(prompt)
print(response)

NameError: name 'tokenizer' is not defined

## <span style="color:red; font-weight:bold;">Using SelfCheckGPT to deduce baseline hallucination of model</span>

In [9]:
!pip install selfcheckgpt


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel
import spacy
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [22]:
# using spaCy for sentence segmentation
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

In [23]:
model_id = "mistralai/Mixtral-8x7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

C:\Users\devanshi\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\devanshi\.cache\huggingface\hub\models--mistralai--Mixtral-8x7B-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some parameters are on the meta device because they were offloaded to the cpu an

In [24]:
# function to generate responses with the model
def generate_responses(prompt, num_samples=3, max_new_tokens=200): # generating multiple responses from Mixtral for the same prompt
    responses = []
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    for _ in range(num_samples):
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,  
                temperature=0.8,
                top_p=0.9,
                repetition_penalty=1.2
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if response.startswith(prompt):
            response = response[len(prompt):].strip()
        responses.append(response)
    
    return responses

In [25]:
# segment text into sentences
def get_sentences(text):
    doc = nlp(text)
    sentences = [sent.text.strip() for sent in doc.sents if len(sent) > 3]
    return sentences

In [26]:
# function to compute BERTScore
def compute_bertscore(sentences, reference_samples):
    """
    Compute BERTScore between sentences and reference samples.
    """
    from selfcheckgpt.modeling_selfcheck import SelfCheckBERTScore
    
    bertscore = SelfCheckBERTScore()
    scores = bertscore.predict(sentences, reference_samples)
    return scores

In [27]:
# function to generate MQAG score against reference samples
def compute_mqag(sentences, main_text, reference_samples, device="cuda"):
    from selfcheckgpt.modeling_selfcheck import SelfCheckMQAG
    
    mqag = SelfCheckMQAG(device=device)
    scores = mqag.predict(
        sentences,
        main_text,
        reference_samples,
        num_questions_per_sent=3,  # Reduced for speed
        scoring_method='bayes_with_alpha',
        beta1=0.95, beta2=0.95,
    )
    return scores

In [28]:
def run_selfcheck(prompt, use_bertscore=True, use_mqag=True, num_samples=3):
    print(f"Generating {num_samples} responses for: {prompt}")

In [30]:
main_response = generate_responses(prompt, num_samples=1)[0]
print(f"\nMain response:\n{main_response}\n")

reference_samples = generate_responses(prompt, num_samples=num_samples)
print(f"Generated {len(reference_samples)} reference samples.")

KeyboardInterrupt: 

In [ ]:
sentences = get_sentences(main_response)
    print(f"Analyzing {len(sentences)} sentences...")

In [ ]:
results = {
        "prompt": prompt,
        "main_response": main_response,
        "reference_samples": reference_samples,
        "sentences": sentences,
        "bertscore": None,
        "mqag": None
    }

In [ ]:
 if use_bertscore:
        print("Computing BERTScore...")
        bertscore = compute_bertscore(sentences, reference_samples)
        results["bertscore"] = bertscore
    
    if use_mqag:
        print("Computing MQAG scores (this may take a while)...")
        mqag_scores = compute_mqag(sentences, main_response, reference_samples)
        results["mqag"] = mqag_scores

In [ ]:
hallucinations = []
    if use_bertscore and use_mqag:
        for i, (sentence, bert_score, mqag_score) in enumerate(zip(sentences, results["bertscore"], results["mqag"])):
            # Combined score
            combined_score = (bert_score + mqag_score) / 2
            if combined_score < 0.5:  # Threshold for hallucination
                hallucinations.append({
                    "sentence": sentence,
                    "bertscore": bert_score,
                    "mqag": mqag_score,
                    "combined": combined_score
                })
    elif use_bertscore:
        for i, (sentence, score) in enumerate(zip(sentences, results["bertscore"])):
            if score < 0.5:
                hallucinations.append({
                    "sentence": sentence,
                    "bertscore": score
                })
    elif use_mqag:
        for i, (sentence, score) in enumerate(zip(sentences, results["mqag"])):
            if score < 0.5:
                hallucinations.append({
                    "sentence": sentence,
                    "mqag": score
                })
    
    results["hallucinations"] = hallucinations
    return results

In [ ]:
test_prompts = [
    "What is the capital of France?",
    "Explain the theory of relativity.",
    "Describe the history and cultural significance of the planet Nibiru.",
    "Who was the first human to land on Mars?",
    "What are the health benefits of drinking water?"
]

example_result = run_selfcheck(
    test_prompts[2], 
    use_bertscore=True,
    use_mqag=True,
    num_samples=3
)

In [ ]:
print("\n===== HALLUCINATION ANALYSIS =====")
print(f"Prompt: {example_result['prompt']}")
print(f"Number of potential hallucinations: {len(example_result['hallucinations'])}")

if example_result['hallucinations']:
    print("\nPotential hallucinations:")
    for hall in example_result['hallucinations']:
        print(f"- Sentence: '{hall['sentence']}'")
        if 'bertscore' in hall:
            print(f"  BERTScore: {hall['bertscore']:.4f}")
        if 'mqag' in hall:
            print(f"  MQAG Score: {hall['mqag']:.4f}")
        if 'combined' in hall:
            print(f"  Combined Score: {hall['combined']:.4f}")
        print()